In [7]:
import pandas as pd
import numpy as np

In [8]:
# ======================================================
# Parameter counters
# ======================================================

def mlp_parameters(
    input_dim,
    hidden_width,
    hidden_layers,
    output_dim=1,
):
    dims = [input_dim] + [hidden_width] * hidden_layers + [output_dim]

    total = 0
    for din, dout in zip(dims[:-1], dims[1:]):
        total += din * dout + dout

    return total


def kan_parameters(
    input_dim,
    hidden_width,
    hidden_layers,
    grid=3,
    spline_order=3,
    output_dim=1,
):
    dims = [input_dim] + [hidden_width] * hidden_layers + [output_dim]

    total = 0

    factor = grid + spline_order + 3

    for din, dout in zip(dims[:-1], dims[1:]):
        total += (din * dout) * factor + dout

    return total

In [9]:
import math
import pandas as pd

# ==========================================================
# Configuration
# ==========================================================

INPUT_DIM = 2
OUTPUT_DIM = 1
NUM_NETWORKS = 2      # u-network + k-network

GRID = 3
SPLINE_ORDER = 3

MLP_WIDTHS = [75, 100, 125]
DEPTHS = [3, 4, 5]


# ==========================================================
# Parameter formulas
# ==========================================================

def mlp_parameters(width, depth):

    dims = [INPUT_DIM] + [width] * depth + [OUTPUT_DIM]

    total = 0

    for din, dout in zip(dims[:-1], dims[1:]):
        total += din * dout + dout

    return NUM_NETWORKS * total


def kan_parameters(width, depth):

    dims = [INPUT_DIM] + [width] * depth + [OUTPUT_DIM]

    total = 0

    factor = GRID + SPLINE_ORDER + 3

    for din, dout in zip(dims[:-1], dims[1:]):
        total += din * dout * factor + dout

    return NUM_NETWORKS * total


# ==========================================================
# FLOP formulas (from the paper)
# ==========================================================

def kan_multiplier():
    """
    FLOP multiplier of one KAN connection.
    """

    G = GRID
    K = SPLINE_ORDER

    return 9 * K * (G + 1.5 * K) + 2 * G - 2.5 * K + 3


KAN_FACTOR = kan_multiplier()


def mlp_flops(width, depth):

    dims = [INPUT_DIM] + [width] * depth + [OUTPUT_DIM]

    total = 0

    for din, dout in zip(dims[:-1], dims[1:]):
        total += 2 * din * dout

    return NUM_NETWORKS * total


def kan_flops(width, depth):

    dims = [INPUT_DIM] + [width] * depth + [OUTPUT_DIM]

    total = 0

    for din, dout in zip(dims[:-1], dims[1:]):
        total += din * dout * KAN_FACTOR

    return NUM_NETWORKS * total


# ==========================================================
# Search equivalent widths
# ==========================================================

KAN_WIDTHS = range(2, 51)

rows = []

for depth in DEPTHS:

    for mlp_width in MLP_WIDTHS:

        p_mlp = mlp_parameters(mlp_width, depth)
        f_mlp = mlp_flops(mlp_width, depth)

        best_param = None
        best_flop = None

        best_param_diff = float("inf")
        best_flop_diff = float("inf")

        for kan_width in KAN_WIDTHS:

            p_kan = kan_parameters(kan_width, depth)
            f_kan = kan_flops(kan_width, depth)

            if abs(p_kan - p_mlp) < best_param_diff:
                best_param_diff = abs(p_kan - p_mlp)

                best_param = (
                    kan_width,
                    p_kan,
                )

            if abs(f_kan - f_mlp) < best_flop_diff:
                best_flop_diff = abs(f_kan - f_mlp)

                best_flop = (
                    kan_width,
                    f_kan,
                )

        rows.append(
            {
                "Layers": depth,

                "MLP width": mlp_width,
                "MLP params": p_mlp,
                "MLP FLOPs": f_mlp,

                "KAN width (params)": best_param[0],
                "KAN params": best_param[1],

                "KAN width (FLOPs)": best_flop[0],
                "KAN FLOPs": best_flop[1],
            }
        )

df = pd.DataFrame(rows)

pd.set_option("display.max_columns", None)

print(df)

df.to_csv("architecture_matching.csv", index=False)

print("\nSaved to architecture_matching.csv")

   Layers  MLP width  MLP params  MLP FLOPs  KAN width (params)  KAN params  \
0       3         75       23402      45900                  25       24002   
1       3        100       41202      81200                  33       41186   
2       3        125       64002     126500                  41       62978   
3       4         75       34802      68400                  25       35302   
4       4        100       61402     121200                  33       60854   
5       4        125       95502     189000                  41       93318   
6       5         75       46202      90900                  25       46602   
7       5        100       81602     161200                  33       80522   
8       5        125      127002     251500                  42      129698   

   KAN width (FLOPs)  KAN FLOPs  
0                  7    48552.0  
1                  9    77112.0  
2                 12   132192.0  
3                  7    68544.0  
4                  9   110160.0  
5    

In [11]:
df

,Layers,MLP width,MLP params,MLP FLOPs,KAN width (params),KAN params,KAN width (FLOPs),KAN FLOPs
0,3,75,23402,45900,25,24002,7,48552.0
1,3,100,41202,81200,33,41186,9,77112.0
2,3,125,64002,126500,41,62978,12,132192.0
3,4,75,34802,68400,25,35302,7,68544.0
4,4,100,61402,121200,33,60854,9,110160.0
5,4,125,95502,189000,41,93318,12,190944.0
6,5,75,46202,90900,25,46602,7,88536.0
7,5,100,81602,161200,33,80522,10,175440.0
8,5,125,127002,251500,42,129698,12,249696.0
